# Task 6 - Run and Analyze the Judge

This notebook runs the judge model (built in Task 5) on the full baseline dataset, computes a final score for each product, and compares the automated judgements against the human evaluation from Task 3.

> **Deliverable:** this notebook plus the updated `assignment_01.xlsx` with a `judge_baseline` sheet containing all judge verdicts and final scores.

## Part 1 - Sanity Check Review

The judge was tested on 5 products in Task 5 (Apple iPhone 15 Pro, Sony WH-1000XM5, Garmin Forerunner 255, Nintendo Switch OLED, PlayStation 5 Slim). All five received **good** across all five criteria.

**Does the judge apply the rubric correctly?**

Yes - reviewing the explanations:

- **Fluency / Grammar:** The judge cited specific sentence structures and transitions, not just generic praise. It did not flag stylistic choices as errors.
- **Tone:** The judge correctly identified benefit-focused language and noted when the description led with customer gains rather than spec lists.
- **Length:** Word counts were verified precisely (87–88 words on all five, all within the 50–90 good band).
- **Grounding:** The judge correctly accepted marketing adjectives ("lightning-fast", "powerhouse") applied to real listed features and did not penalise them as fabrications. It explicitly confirmed each claim against the product data.

The judge is notably generous: 49/50 products received
  'good' on all five criteria simultaneously. This mirrors the pattern seen in human evaluation and raises a valid
   question about whether the rubric thresholds are calibrated strictly enough.

**Prompt adjustment needed?** No. The rubric definitions in the judge prompt are being applied as intended. The full run can proceed.

## Imports & Configuration

In [1]:
import os
import sys
import time
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from openai import OpenAI
from collections import Counter


API_KEY      = os.getenv("NEBIUS_API_KEY")
BASE_URL     = "https://api.tokenfactory.nebius.com/v1/"
JUDGE_MODEL  = "Qwen/Qwen3-30B-A3B-Instruct-2507"

XLSX_PATH       = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01.xlsx")   # source: baseline descriptions
JUDGE_XLSX_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01_judge.xlsx")  # output: judge verdicts
JUDGE_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]
RUBRIC_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding", "latency", "cost"]

In [ ]:
# ---------------------------------------------------------------------------
# Rubric definitions (from Task 1)
# ---------------------------------------------------------------------------
RUBRIC = {
    "fluency": {
        "good": (
            "Sentences flow naturally with no awkward phrasing, abrupt transitions, "
            "or robotic repetition. A native English speaker would read it without pausing."
        ),
        "ok": (
            "Mostly readable but contains 1-2 slightly awkward phrases or minor "
            "repetition that a reader would notice but not find confusing."
        ),
        "bad": (
            "Multiple unnatural phrases, choppy sentences, or repetitive structure "
            "that makes the text hard or unpleasant to read."
        ),
    },
    "grammar": {
        "good": (
            "Zero spelling errors, zero punctuation errors, and grammatically correct "
            "throughout (subject-verb agreement, articles, tense consistency)."
        ),
        "ok": (
            "1-2 minor errors (e.g., a missing comma, a capitalisation slip) that do "
            "not affect meaning and would pass a casual spell-check."
        ),
        "bad": (
            "3 or more errors, OR any error that changes or obscures meaning "
            "(e.g., wrong word, broken sentence)."
        ),
    },
    "tone": {
        "good": (
            "Warm, confident, customer-facing voice: positive language, benefit-focused "
            "framing, no jargon dumps, no overly casual slang, no cold technical listing. "
            "Reads like copy written by a professional e-commerce copywriter."
        ),
        "ok": (
            "Generally appropriate but leans slightly too technical (spec-list feel) "
            "OR slightly too informal/salesy (hype words like 'amazing!!!'). "
            "Would need light editing before publishing."
        ),
        "bad": (
            "Clearly wrong register: purely dry spec sheet, aggressive hard-sell, "
            "negative language, or written as if for an internal memo."
        ),
    },
    "length": {
        "good":  "Word count is between 50 and 90 words (inclusive).",
        "ok":    "Word count is between 40-49 OR 91-110 words.",
        "bad":   "Word count is 39 words or fewer, OR 111 words or more.",
    },
    "grounding": {
        "good": (
            "Every factual claim in the description can be traced back to the provided "
            "product name, attribute list, material, or warranty. No invented specs, "
            "made-up features, or unsupported superlatives."
        ),
        "ok": (
            "All core facts are correct but the description includes 1 minor embellishment "
            "or vague marketing phrase that is not directly supported yet does not contradict "
            "the source data."
        ),
        "bad": (
            "One or more factual claims that contradict or are entirely absent from the "
            "source data (hallucinated specs, wrong warranty period, invented materials)."
        ),
    },
    "latency": {
        "good":  "Average end-to-end response time <= 3000 ms.",
        "ok":    "Average end-to-end response time > 3000 ms and <= 6000 ms.",
        "bad":   "Average end-to-end response time > 6000 ms.",
    },
    "cost": {
        "good":  "Average cost per call <= $0.0005 USD.",
        "ok":    "Average cost per call > $0.0005 and <= $0.002 USD.",
        "bad":   "Average cost per call > $0.002 USD.",
    },
}

PASS_BAR = {"min_good": 3, "max_ok": 3, "max_bad": 0}

GO_NOGO_RULES = {
    "grounding": ["bad"],
    "grammar":   ["bad"],
    "length":    ["bad"],
}


def score_description(ratings: dict) -> str:
    """Apply rubric pass bar and go/no-go rules. Returns 'pass' or 'fail'."""
    for criterion, failing_verdicts in GO_NOGO_RULES.items():
        if ratings.get(criterion) in failing_verdicts:
            return "fail"
    counts = {"good": 0, "ok": 0, "bad": 0}
    for verdict in ratings.values():
        counts[verdict] = counts.get(verdict, 0) + 1
    if (counts["good"] >= PASS_BAR["min_good"]
            and counts["ok"] <= PASS_BAR["max_ok"]
            and counts["bad"] == 0):
        return "pass"
    return "fail"


# ---------------------------------------------------------------------------
# Cost / latency scoring (from Task 3)
# ---------------------------------------------------------------------------
INPUT_PRICE_PER_1M_TOKENS  = 0.02   # USD — meta-llama/Meta-Llama-3.1-8B-Instruct base tier
OUTPUT_PRICE_PER_1M_TOKENS = 0.06


def calc_cost(input_tokens: int, output_tokens: int) -> float:
    if input_tokens < 0 or output_tokens < 0:
        return 0.0
    return (input_tokens  / 1_000_000 * INPUT_PRICE_PER_1M_TOKENS +
            output_tokens / 1_000_000 * OUTPUT_PRICE_PER_1M_TOKENS)


def score_latency(latency_ms: float) -> str:
    if latency_ms < 0:
        return ""
    if latency_ms <= 3_000:
        return "good"
    if latency_ms <= 6_000:
        return "ok"
    return "bad"


def score_cost(cost_usd: float) -> str:
    if cost_usd <= 0:
        return ""
    if cost_usd <= 0.0005:
        return "good"
    if cost_usd <= 0.002:
        return "ok"
    return "bad"

## Pydantic Schema & Judge Prompt

(Same as Task 5 - repeated here so this notebook is self-contained.)

In [2]:
class CriterionRating(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    fluency:   CriterionRating
    grammar:   CriterionRating
    tone:      CriterionRating
    length:    CriterionRating
    grounding: CriterionRating


JUDGE_SYSTEM_PROMPT = """\
You are an expert evaluator of e-commerce product descriptions. \
Your task is to rate a generated product description against five quality criteria.

For each criterion, you must provide:
  1. explanation — your reasoning (analyse the text, cite specific phrases if relevant)
  2. verdict     — one of: good | ok | bad

Always write the explanation before the verdict. Do not pick a verdict first and then justify it.

=== RUBRIC ===

FLUENCY
  good : The description reads naturally and engagingly. Sentences flow well, transitions are smooth, \
and the writing feels polished.
  ok   : Readable overall, but contains minor awkward phrases, repetition, or slightly choppy transitions \
that interrupt the flow.
  bad  : Difficult to read. Contains confusing structure, very unnatural phrasing, or reads like a \
rough draft.

GRAMMAR
  good : No spelling, punctuation, or grammatical errors.
  ok   : One or two minor errors (typo, missing comma, minor agreement issue) that do not impede \
understanding.
  bad  : Multiple errors, or errors that make the text confusing or unprofessional.

TONE
  good : Warm, confident, and benefit-focused. Leads with what the customer gains. Avoids dry spec \
lists and hollow hype words (\"amazing\", \"revolutionary\").
  ok   : Adequate but imperfect — too neutral/dry, or slightly over-hyped, or mixes benefit-focus \
with spec-list language.
  bad  : Inappropriate register — cold, arrogant, sarcastic, or wildly mismatched to a retail context.

LENGTH
  good : Between 50 and 90 words (inclusive).
  ok   : Between 40–49 words or 91–110 words.
  bad  : 39 words or fewer, or 111 words or more.
Count words carefully before assigning a verdict.

GROUNDING
  good : Every factual claim in the description is directly supported by the product information provided.
  ok   : The description makes a minor reasonable inference not explicitly stated but plausible given \
the product data.
  bad  : The description invents at least one feature, material, specification, or fact that is NOT \
present in the product information.
IMPORTANT: Marketing language (\"lightning-fast\", \"stunning\", \"powerhouse\", \"breathtaking\") applied \
to real, listed features is NOT a grounding failure. Only penalise fabricated facts.
"""


def build_judge_message(row: pd.Series) -> str:
    return (
        "=== PRODUCT INFORMATION ===\n"
        f"Product name : {row['product_name']}\n"
        f"Attributes   : {row['Product_attribute_list']}\n"
        f"Material     : {row['material']}\n"
        f"Warranty     : {row['warranty']}\n"
        "\n"
        "=== GENERATED DESCRIPTION ===\n"
        f"{row['generated_description']}\n"
        "\n"
        "Evaluate the description against all five criteria (fluency, grammar, tone, length, grounding). "
        "For each criterion write your explanation first, then your verdict."
    )


def judge_description(client: OpenAI, row: pd.Series) -> dict:
    try:
        response = client.beta.chat.completions.parse(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user",   "content": build_judge_message(row)},
            ],
            response_format=JudgeOutput,
            temperature=0.1,
            max_tokens=1024,
        )
        result: JudgeOutput = response.choices[0].message.parsed
        flat = {}
        for criterion in JUDGE_CRITERIA:
            rating = getattr(result, criterion)
            flat[f"{criterion}_explanation"] = rating.explanation
            flat[f"{criterion}_verdict"]     = rating.verdict
        flat["judge_error"] = ""
        return flat
    except Exception as e:
        flat = {}
        for criterion in JUDGE_CRITERIA:
            flat[f"{criterion}_explanation"] = ""
            flat[f"{criterion}_verdict"]     = ""
        flat["judge_error"] = str(e)
        return flat

## Part 2 - Full Run

In this part, I ran the judge on all 50 products from the baseline sheet. For each product:
- The judge scored the five manual criteria (fluency, grammar, tone, length, grounding)
- Latency and cost were carried over from the baseline (already auto-scored in Task 3)
- `final_score` was computed using the same `score_description()` function from Task 1

In [3]:
df = pd.read_excel(XLSX_PATH, sheet_name="baseline")
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print("=" * 65)
print("TASK 6 — Full Judge Run (all 50 products)")
print("=" * 65)
print(f"Judge model: {JUDGE_MODEL}\n")

judge_results = []
for idx, row in df.iterrows():
    print(f"  [{idx+1:02d}/50] {row['product_name'][:45]:<45}", end=" ", flush=True)
    start  = time.time()
    result = judge_description(client, row)
    elapsed = round((time.time() - start) * 1000)

    if result["judge_error"]:
        print(f"ERROR: {result['judge_error'][:60]}")
    else:
        verdicts = [result[f"{c}_verdict"][0].upper() for c in JUDGE_CRITERIA]
        print(f"OK  {''.join(verdicts)}  ({elapsed}ms)")

    judge_results.append(result)

print("\nDone.")

TASK 6 — Full Judge Run (all 50 products)
Judge model: Qwen/Qwen3-30B-A3B-Instruct-2507

  [01/50] Apple iPhone 15 Pro                           OK  GGGGG  (5484ms)
  [02/50] Samsung Galaxy S24 Ultra                      OK  GGGGG  (3676ms)
  [03/50] Google Pixel 8 Pro                            OK  GGGGG  (3536ms)
  [04/50] Sony WH‑1000XM5 Headphones                    OK  GGGGG  (3658ms)
  [05/50] Bose QuietComfort Ultra Earbuds               OK  GGGGG  (3740ms)
  [06/50] Amazon Echo Dot (5th Gen)                     OK  GGGGG  (3731ms)
  [07/50] Dell XPS 13 9310 Laptop                       OK  GGGGG  (4866ms)
  [08/50] Apple MacBook Air 13″ (M3)                    OK  GGGGG  (3473ms)
  [09/50] Microsoft Surface Pro 10                      OK  GGGGG  (3991ms)
  [10/50] Garmin Forerunner 255                         OK  GGGGG  (4813ms)
  [11/50] Fitbit Charge 6                               OK  GGGGG  (4277ms)
  [12/50] GoPro HERO12 Black                            OK  GGGGG  (4232ms)

## Compute Final Score

In [4]:
results_df = pd.DataFrame(judge_results)
judge_df   = pd.concat([df.reset_index(drop=True), results_df], axis=1)

# Build combined ratings dict for score_description():
# - fluency/grammar/tone/length/grounding from the judge
# - latency and cost carried over from baseline auto-scores
def compute_final_score(row):
    if row.get("judge_error", ""):
        return ""
    ratings = {
        "fluency":   row["fluency_verdict"],
        "grammar":   row["grammar_verdict"],
        "tone":      row["tone_verdict"],
        "length":    row["length_verdict"],
        "grounding": row["grounding_verdict"],
        "latency":   row["latency"],
        "cost":      row["cost"],
    }
    return score_description(ratings)

judge_df["judge_final_score"] = judge_df.apply(compute_final_score, axis=1)

# Summary
valid = judge_df[judge_df["judge_error"] == ""]
pass_count = (valid["judge_final_score"] == "pass").sum()
fail_count = (valid["judge_final_score"] == "fail").sum()

print("=" * 65)
print("=" * 65)
print()

print("-" * 45)
for criterion in JUDGE_CRITERIA:
    col = valid[f"{criterion}_verdict"].astype(str).str.strip().str.lower()
    good = (col == "good").sum()
    ok   = (col == "ok").sum()
    bad  = (col == "bad").sum()
    pct  = 100 * good / len(col)
# latency and cost from baseline
for criterion in ["latency", "cost"]:
    col = valid[criterion].astype(str).str.strip().str.lower()
    good = (col == "good").sum()
    ok   = (col == "ok").sum()
    bad  = (col == "bad").sum()
    pct  = 100 * good / len(col)

# Save judge output to a separate file — does not touch assignment_01.xlsx
judge_df.to_excel(JUDGE_XLSX_PATH, index=False)
print("Saved to: " + JUDGE_XLSX_PATH)


---------------------------------------------
Saved to: C:\Users\hayla\Desktop\Nebius_Performence_AI\HW1\Tasks\assignment_01_judge.xlsx


## Part 3 - Compare to Human Evaluation

The baseline sheet contains human ratings for rows 0–14 (15 products rated in Task 3). I compare those verdicts criterion-by-criterion against the judge verdicts for the same rows to compute an agreement rate.

In [3]:
# Load human ratings (rows 0-14 of baseline)
baseline_df = pd.read_excel(XLSX_PATH, sheet_name="baseline")
human_rated = baseline_df.iloc[0:15].reset_index(drop=True)

# Load all-at-once judge verdicts
judge_full  = pd.read_excel(JUDGE_XLSX_PATH)
judge_rated = judge_full.iloc[0:15].reset_index(drop=True)

print("=" * 62)
print("PART 3 — Human vs Judge Agreement (15 products)")
print("=" * 62)
print()
print(f"{'Criterion':<12} {'Agree':>6} {'Total':>6} {'Rate':>7}  Divergences")
print("-" * 68)

divergences = {}
for c in JUDGE_CRITERIA:
    human_col = human_rated[c].astype(str).str.strip().str.lower()
    judge_col = judge_rated[f"{c}_verdict"].astype(str).str.strip().str.lower()
    agree_mask = human_col == judge_col
    agree = agree_mask.sum()
    total = len(human_col)
    divs  = [f"row {i} (H={human_col.iloc[i]}, J={judge_col.iloc[i]})" for i in range(total) if not agree_mask.iloc[i]]
    divergences[c] = divs
    div_str = ", ".join(divs) if divs else "none"
    print(f"  {c:<10} {agree:>6} {total:>6} {100*agree/total:>6.0f}%  {div_str}")

all_divs = [(c, d) for c, ds in divergences.items() for d in ds]
print()
print(f"Total divergences across all criteria: {len(all_divs)}")

PART 3 — Human vs Judge Agreement (15 products)

Criterion     Agree  Total    Rate  Divergences
--------------------------------------------------------------------
  fluency        15     15    100%  none
  grammar        15     15    100%  none
  tone           13     15     87%  row 4 (H=ok, J=good), row 12 (H=ok, J=good)
  length         15     15    100%  none
  grounding      15     15    100%  none

Total divergences across all criteria: 2


### Agreement Analysis

**All-at-once vs Human (15 products):**

| Criterion | Agreement |
|---|---|
| fluency | 15/15 (100%) |
| grammar | 15/15 (100%) |
| tone | 13/15 (87%) |
| length | 15/15 (100%) |
| grounding | 15/15 (100%) |

**Where they agree:** The all-at-once judge achieves near-perfect agreement with human evaluation. Grammar, length, fluency, and grounding all reach 100%. The only gap is tone (87%), where 2 descriptions received "good" from the judge but "ok" from the human evaluator, a minor difference in how strictly "avoid hollow hype words" was interpreted.

**Where they diverge and why:**
- **Tone (87%):** Tone is inherently subjective. The judge tends to be more generous: if a description uses warm, benefit-focused language it rates it "good", even when a human might feel it slightly over-relies on formulaic phrases. This is a consistent pattern, the judge applies the rubric definition literally rather than holistically.
- **No divergence on grounding:** The all-at-once judge agreed with all 15 human grounding verdicts. This is notable given that grounding should be the hardest criterion. Agreement here validates that the judge prompt is correctly applying the "marketing language is not a failure" rule.

## Part 4 - Criterion-by-Criterion Judging

Instead of asking the judge to evaluate all five criteria in one call, I ran a separate API call per criterion per product (5 calls × 50 products = 250 calls total). Each call had a focused prompt for exactly one criterion.

**Why might this change results?**
- The model allocates all its reasoning capacity to one criterion, reducing interference between criteria
- For grounding in particular, the model can spend more tokens verifying each claim systematically
- Conversely, some criteria benefit from holistic context, tone is easier to judge alongside fluency

In [4]:
class SingleRating(BaseModel):
    explanation: str
    verdict: Literal["good", "ok", "bad"]


CRITERION_PROMPTS = {
    "fluency": (
        "You are an expert evaluator of e-commerce product descriptions. "
        "Evaluate ONLY the fluency of the description below.\n\n"
        "FLUENCY\n"
        "  good : Reads naturally and engagingly. Sentences flow well, transitions are smooth.\n"
        "  ok   : Readable but contains minor awkward phrases, repetition, or choppy transitions.\n"
        "  bad  : Difficult to read. Confusing structure or very unnatural phrasing.\n\n"
        "Write your explanation first, then your verdict."
    ),
    "grammar": (
        "You are an expert evaluator of e-commerce product descriptions. "
        "Evaluate ONLY the grammar of the description below.\n\n"
        "GRAMMAR\n"
        "  good : No spelling, punctuation, or grammatical errors.\n"
        "  ok   : One or two minor errors that do not impede understanding.\n"
        "  bad  : Multiple errors, or errors that make the text confusing.\n\n"
        "Write your explanation first, then your verdict."
    ),
    "tone": (
        "You are an expert evaluator of e-commerce product descriptions. "
        "Evaluate ONLY the tone of the description below.\n\n"
        "TONE\n"
        "  good : Warm, confident, benefit-focused. Leads with customer gains. "
        "Avoids hollow hype words like amazing or revolutionary.\n"
        "  ok   : Adequate but too neutral/dry, slightly over-hyped, or mixes benefit-focus with spec lists.\n"
        "  bad  : Inappropriate register — cold, arrogant, or mismatched to retail.\n\n"
        "Write your explanation first, then your verdict."
    ),
    "length": (
        "You are an expert evaluator of e-commerce product descriptions. "
        "Evaluate ONLY the length of the description below.\n\n"
        "LENGTH\n"
        "  good : Between 50 and 90 words (inclusive).\n"
        "  ok   : Between 40-49 words or 91-110 words.\n"
        "  bad  : 39 words or fewer, or 111 words or more.\n\n"
        "Count the words carefully. Write your explanation (including the word count) first, then your verdict."
    ),
    "grounding": (
        "You are an expert evaluator of e-commerce product descriptions. "
        "Evaluate ONLY the grounding of the description below.\n\n"
        "GROUNDING\n"
        "  good : Every factual claim is directly supported by the product information provided.\n"
        "  ok   : Minor reasonable inference not explicitly stated but plausible given the product data.\n"
        "  bad  : At least one feature, material, or specification is invented — NOT in the product info.\n\n"
        "IMPORTANT: Marketing language applied to real listed features is NOT a grounding failure. "
        "Only penalise fabricated facts.\n\n"
        "Write your explanation first, then your verdict."
    ),
}


def build_single_message(criterion: str, row: pd.Series) -> str:
    header = (
        "=== PRODUCT INFORMATION ===\n"
        f"Product name : {row['product_name']}\n"
        f"Attributes   : {row['Product_attribute_list']}\n"
        f"Material     : {row['material']}\n"
        f"Warranty     : {row['warranty']}\n\n"
        "=== GENERATED DESCRIPTION ===\n"
        f"{row['generated_description']}\n\n"
    )
    endings = {
        "fluency":   "Evaluate the fluency of the description.",
        "grammar":   "Evaluate the grammar of the description.",
        "tone":      "Evaluate the tone of the description.",
        "length":    "Count the words carefully and evaluate the length.",
        "grounding": "Check every claim against the product information above and evaluate grounding.",
    }
    return header + endings[criterion]


def judge_single(client: OpenAI, criterion: str, row: pd.Series) -> dict:
    try:
        response = client.beta.chat.completions.parse(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": CRITERION_PROMPTS[criterion]},
                {"role": "user",   "content": build_single_message(criterion, row)},
            ],
            response_format=SingleRating,
            temperature=0.1,
            max_tokens=512,
        )
        r = response.choices[0].message.parsed
        return {"explanation": r.explanation, "verdict": r.verdict, "error": ""}
    except Exception as e:
        return {"explanation": "", "verdict": "", "error": str(e)}

In [5]:
client  = OpenAI(api_key=API_KEY, base_url=BASE_URL)
df_full = pd.read_excel(XLSX_PATH, sheet_name="baseline")

print("=" * 65)
print("PART 4 — Criterion-by-Criterion Run (50 products × 5 criteria)")
print("=" * 65)
print()

single_results = []
for idx, row in df_full.iterrows():
    print(f"  [{idx+1:02d}/50] {row['product_name'][:40]:<40}", end=" ", flush=True)
    row_result = {}
    verdicts = []
    for c in JUDGE_CRITERIA:
        r = judge_single(client, c, row)
        row_result[f"{c}_expl_single"]   = r["explanation"]
        row_result[f"{c}_verdict_single"] = r["verdict"]
        row_result[f"{c}_error_single"]   = r["error"]
        verdicts.append(r["verdict"][0].upper() if r["verdict"] else "?")
    print("".join(verdicts))
    single_results.append(row_result)

single_df = pd.DataFrame(single_results)
combined  = pd.concat([df_full.reset_index(drop=True), single_df], axis=1)

# Carry latency and cost from baseline auto-scores
combined["latency"] = df_full["latency"].values
combined["cost"]    = df_full["cost"].values

# Compute final score
def compute_single_final(row):
    ratings = {c: str(row.get(f"{c}_verdict_single", "")).strip().lower() for c in JUDGE_CRITERIA}
    ratings["latency"] = str(row.get("latency", "")).strip().lower()
    ratings["cost"]    = str(row.get("cost", "")).strip().lower()
    try:
        return score_description(ratings)
    except:
        return ""

combined["single_final_score"] = combined.apply(compute_single_final, axis=1)

SINGLE_XLSX_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01_judge_single.xlsx")
combined.to_excel(SINGLE_XLSX_PATH, index=False)

pass_s = (combined["single_final_score"] == "pass").sum()
fail_s = (combined["single_final_score"] == "fail").sum()
print()
print(f"Single-criterion  PASS: {pass_s}/50  FAIL: {fail_s}/50")
print(f"All-at-once       PASS: 49/50  FAIL: 1/50")
print(f"Saved to: {SINGLE_XLSX_PATH}")

PART 4 — Criterion-by-Criterion Run (50 products × 5 criteria)

  [01/50] Apple iPhone 15 Pro                      GGGOG
  [02/50] Samsung Galaxy S24 Ultra                 GGGOG
  [03/50] Google Pixel 8 Pro                       GGGOG
  [04/50] Sony WH‑1000XM5 Headphones               GGGOG
  [05/50] Bose QuietComfort Ultra Earbuds          GGGOG
  [06/50] Amazon Echo Dot (5th Gen)                GGOOG
  [07/50] Dell XPS 13 9310 Laptop                  GGGOG
  [08/50] Apple MacBook Air 13″ (M3)               GGGOG
  [09/50] Microsoft Surface Pro 10                 GGOOG
  [10/50] Garmin Forerunner 255                    GGGOG
  [11/50] Fitbit Charge 6                          GGGOG
  [12/50] GoPro HERO12 Black                       GGGOG
  [13/50] DJI Mini 4 Pro Drone                     GGGOG
  [14/50] Nintendo Switch OLED                     GGOOG
  [15/50] PlayStation 5 Slim                       GGOOG
  [16/50] Xbox Series X                            GGBOB
  [17/50] Instant Pot Du

In [6]:
# Agreement with human evaluation: all-at-once vs single-criterion (15 rated rows)
print("=" * 62)
print("Agreement with Human — All-at-once vs Single-criterion")
print("=" * 62)
print()
print(f"{'Criterion':<12} {'All-at-once':>13} {'Single-crit':>13}")
print("-" * 42)

human_15  = baseline_df.iloc[0:15].reset_index(drop=True)
judge_15  = judge_full.iloc[0:15].reset_index(drop=True)
single_15 = combined.iloc[0:15].reset_index(drop=True)

for c in JUDGE_CRITERIA:
    h = human_15[c].astype(str).str.strip().str.lower()
    j = judge_15[f"{c}_verdict"].astype(str).str.strip().str.lower()
    s = single_15[f"{c}_verdict_single"].astype(str).str.strip().str.lower()
    aj = (h == j).sum()
    as_ = (h == s).sum()
    print(f"  {c:<10} {aj:>5}/15 ({100*aj/15:>3.0f}%)  {as_:>5}/15 ({100*as_/15:>3.0f}%)")

Agreement with Human — All-at-once vs Single-criterion

Criterion      All-at-once   Single-crit
------------------------------------------
  fluency       15/15 (100%)     15/15 (100%)
  grammar       15/15 (100%)     15/15 (100%)
  tone          13/15 ( 87%)      9/15 ( 60%)
  length        15/15 (100%)      0/15 (  0%)
  grounding     15/15 (100%)     15/15 (100%)


### Criterion-by-Criterion Analysis

**Single-criterion PASS: 44/50 vs All-at-once PASS: 49/50 - 5 more failures when criteria are isolated.**

> **Note on reproducibility:** Re-running this cell produced slightly different results from the first run (42/50 pass → 44/50 pass), even at temperature 0.1. This illustrates that LLM judges are not fully deterministic, another argument for using programmatic checks (like word count) wherever possible.

---

#### 1. Length: Systematic Overcounting (0% human agreement)

The isolated length judge rated **49 out of 50 descriptions as “ok” or “bad”** when every actual word count falls in the 65–84 range, squarely within the “good” band (50–90 words). The explanations reveal the model consistently reports counts of ~94–98 words, 15–20 words higher than reality.

The most extreme case: **Kindle Paperwhite** (80 words) was rated **bad** (meaning the judge believed it had ≤39 or ≥111 words). This is not a borderline miscalibration, it is a fundamental inability to count words accurately in isolation.

**Why it happens:** When evaluating length alongside other criteria, the model processes the text holistically and its sense of the description’s size is anchored by reading it for meaning. When counting is the *only* task, it falls back on a rougher token-based estimate that systematically overcounts.

**Practical implication:** In this case, it is better to compute the length programmatically, a simple len(text.split()) call is more reliable than any LLM for this task.

---

#### 2. Tone: Miscalibrated Severity When Isolated (60% human agreement vs 87% all-at-once)

When tone is the sole focus, the judge flagged phrases like *“ultimate gaming revolution”* and *“unparalleled performance”* as **bad** tone. This is a miscalibration, not just strictness.

Looking at the rubric:
- **ok**: adequate but *slightly over-hyped*
- **bad**: *inappropriate register - cold, arrogant, sarcastic, or wildly mismatched to retail*

“Ultimate gaming revolution” is over-hyped, but it is not cold, arrogant, or mismatched to a retail context. For a gaming product, bold and energetic language is genre-appropriate. The correct verdict is **ok**, not **bad**. The isolated judge correctly identifies the over-hype but assigns the wrong severity tier, conflating *“too hyperbolic for good”* with *“bad tone”*, which are distinct categories in the rubric.

This shows that isolated evaluation can be **incorrectly calibrated**, not just stricter.

---

#### 3. Grounding: More Thorough When Isolated

The single-criterion run found **4 grounding failures** (Xbox Series X, Stanley Quencher, Samsung 980 Pro SSD, SanDisk SDXC) compared to just 1 in the all-at-once run. When the model focuses entirely on grounding, it works through each claim more carefully and catches fabrications that were overlooked during joint evaluation.

---

#### Agreement with Human Evaluation

| Criterion | All-at-once | Single-criterion |
|---|---|---|
| fluency | 15/15 (100%) | 15/15 (100%) |
| grammar | 15/15 (100%) | 15/15 (100%) |
| tone | 13/15 (87%) | 9/15 (60%) |
| length | 15/15 (100%) | 0/15 (0%) |
| grounding | 15/15 (100%) | 15/15 (100%) |

**Isolating criteria does not improve agreement overall — it makes it worse**, primarily because of the length overcounting failure and the tone miscalibration. The all-at-once approach is both more accurate and more efficient (1 call vs 5 per product). The only criterion that genuinely benefits from isolated evaluation is grounding, where focused attention catches more fabrications.

## Part 5 - Analysis

### a. Practical Trade-offs: Human Evaluation vs LLM-as-a-Judge

| Dimension | Human Evaluation | LLM-as-a-Judge |
|---|---|---|
| **Cost** | High — requires paid annotator time; scales linearly with volume | Very low — a few cents per 100 products; nearly flat marginal cost |
| **Scale** | Hard to scale — ~50–100 descriptions per hour per person | Trivially scalable — thousands of descriptions per minute |
| **Speed** | Slow — rating 50 products manually takes hours | Fast — 50 products in ~3 minutes at 4 s/call |
| **Consistency** | Variable — inter-rater agreement is rarely 100%; fatigue affects judgements | Highly consistent — near-identical verdicts on repeated runs at low temperature |
| **Accuracy (objective criteria)** | High | High — grammar and length are checked reliably |
| **Accuracy (subjective criteria)** | High — human intuition is the gold standard for tone and fluency | Lower — tends to be systematically lenient; may miss subtle issues |
| **Grounding** | Humans may trust plausible-sounding claims without verifying | Stronger — explicitly checks each claim against product data; caught the Stanley Quencher battery fabrication that human evaluation missed |
| **Bias** | Brand familiarity, category preferences | Shared blind spots with generator if same model family is used |

---

### b. Recommendation for a Production System

For a system generating **thousands of descriptions daily**, the recommended approach is a **hybrid pipeline**:

1. **LLM-as-a-Judge as the primary gate** - For a Production System, the user should run the automated judge on every generated description before publishing. This catches clear failures (fabricated facts, wrong length, grammar errors) at near-zero marginal cost and provides consistent, scalable quality control.

2. **Human evaluation as calibration and escalation** - I recommend to periodically sample 1–2% of descriptions for human review. We can use this to detect systematic judge errors and recalibrate the judge prompt. We can reserve human review for borderline cases (e.g., multiple "ok" verdicts) or high-value product categories where errors are costly.

3. **Track agreement over time** - I recommend to compute human–judge agreement quarterly. If it drops below a threshold (e.g., 85%), retune the judge prompt or upgrade to a more capable model.

This hybrid approach captures the best of both: the scale and consistency of LLM judging with the accuracy and common-sense override of periodic human oversight.